# Cattle Weight Estimation — Step 8: Camera Validation Pipeline

This notebook validates each pipeline stage independently using Kaggle test images.
It also generates the Android inference helper code (Kotlin snippets).

Pipeline stages:
1. Cow detection (YOLO/SSD)
2. Completeness check (bbox edge test)
3. Viewpoint classification (side vs. not-side)
4. Blur detection (Laplacian)
5. Distance check (bbox area fraction)
6. On-device segmentation
7. Weight regression (TFLite)
8. Confidence interval display

In [ ]:
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)
# tensorflow pre-installed on Kaggle (2.16+); don't pin 2.15 (Python 3.12 incompatible)
pip('ultralytics', 'opencv-python-headless', 'Pillow', 'numpy', 'matplotlib', 'tqdm')

import os, json, time
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm
# TF 2.16+ does not support P100 (sm_60 arch) GPU on Python 3.12 — run on CPU
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
import tensorflow as tf
print('TF', tf.__version__, '| GPUs visible:', tf.config.list_physical_devices('GPU'))
import pandas as pd

WORK = Path('/kaggle/working')
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

# ── Robust dataset root discovery ─────────────────────────────────────────────
INPUT = Path('/kaggle/input')
print('=== /kaggle/input contents ===')
for entry in sorted(INPUT.iterdir()):
    try:
        n = sum(1 for _ in entry.rglob('*') if _.is_file())
        print(f'  {entry.name}/  ({n} files)')
    except Exception as e:
        print(f'  {entry.name}/  (error: {e})')

def _count_images(p):
    try:
        return sum(1 for f in p.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXTS)
    except:
        return 0

EXPECTED = INPUT / 'cattle-weight-detection-model-dataset-12k'
if EXPECTED.exists() and _count_images(EXPECTED) > 0:
    ROOT = EXPECTED
else:
    candidates = sorted(
        [(d, _count_images(d)) for d in INPUT.iterdir() if d.is_dir()],
        key=lambda x: x[1], reverse=True
    )
    ROOT = candidates[0][0] if candidates else EXPECTED
    print(f'\n[WARN] Expected path empty — auto-selected: {ROOT}')

print(f'\nDataset ROOT : {ROOT}')
print(f'Images found : {_count_images(ROOT)}')

img_files = [f for f in ROOT.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXTS]
print(f'Images loaded into list: {len(img_files)}')

## Stage 1: Cow Detector (YOLOv8n)

In [ ]:
from ultralytics import YOLO

# Use COCO-pretrained YOLOv8n — cow is COCO class 19
detector = YOLO('yolov8n.pt')
COW_CLASS_ID = 19  # COCO
DETECT_CONF = 0.40

def detect_cows(img_bgr, conf=DETECT_CONF):
    """Returns list of (x1,y1,x2,y2,score) for cow detections."""
    results = detector.predict(img_bgr, classes=[COW_CLASS_ID],
                                conf=conf, verbose=False)[0]
    boxes = []
    for box in results.boxes:
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        score = float(box.conf[0])
        boxes.append((x1, y1, x2, y2, score))
    return boxes

# Test on 5 sample images
for p in img_files[:5]:
    img = cv2.imread(str(p))
    if img is None: continue
    dets = detect_cows(img)
    print(f'{p.name}: {len(dets)} cow(s) detected')
    for d in dets:
        print(f'  bbox={d[:4]}  conf={d[4]:.2f}')

In [ ]:
# ── Evaluate detection on sample set ─────────────────────────────────────────
N_EVAL = min(200, len(img_files))
det_results = []
for p in tqdm(img_files[:N_EVAL], desc='Cow detection eval'):
    img = cv2.imread(str(p))
    if img is None:
        det_results.append({'path': str(p), 'n_cows': -1})
        continue
    dets = detect_cows(img)
    det_results.append({'path': str(p), 'n_cows': len(dets),
                         'max_conf': max((d[4] for d in dets), default=0)})

import pandas as pd
det_df = pd.DataFrame(det_results)
print('Detection distribution:')
print(det_df['n_cows'].value_counts().sort_index().to_string())
print(f'\n0 cows detected (false negatives?): {(det_df["n_cows"]==0).mean()*100:.1f}%')
print(f'1 cow (ideal): {(det_df["n_cows"]==1).mean()*100:.1f}%')
print(f'>1 cow:  {(det_df["n_cows"]>1).mean()*100:.1f}%')

## Stage 2 & 5: Completeness and Distance Checks

In [ ]:
def check_completeness(bbox, img_shape, edge_margin=0.03):
    """Returns (ok, reason). Cow must be fully inside frame with margin."""
    x1, y1, x2, y2 = bbox[:4]
    H, W = img_shape[:2]
    margin_px_x = int(W * edge_margin)
    margin_px_y = int(H * edge_margin)
    if x1 < margin_px_x:  return False, 'Cow touching left edge'
    if y1 < margin_px_y:  return False, 'Cow touching top edge'
    if x2 > W - margin_px_x: return False, 'Cow touching right edge'
    if y2 > H - margin_px_y: return False, 'Cow touching bottom edge'
    return True, 'OK'

def check_distance(bbox, img_shape,
                    min_area_frac=0.15, max_area_frac=0.75):
    """Returns (ok, reason). Bbox area should be 15-75% of frame."""
    x1, y1, x2, y2 = bbox[:4]
    H, W = img_shape[:2]
    bbox_area = (x2 - x1) * (y2 - y1)
    frame_area = H * W
    frac = bbox_area / frame_area
    if frac < min_area_frac:
        return False, f'Too far ({frac*100:.0f}% < {min_area_frac*100:.0f}%) — move closer'
    if frac > max_area_frac:
        return False, f'Too close ({frac*100:.0f}% > {max_area_frac*100:.0f}%) — move back'
    return True, f'Good distance ({frac*100:.0f}%)'

# Test
for p in img_files[:5]:
    img = cv2.imread(str(p))
    if img is None: continue
    dets = detect_cows(img)
    if not dets: continue
    bbox = dets[0]  # best detection
    ok_c, msg_c = check_completeness(bbox, img.shape)
    ok_d, msg_d = check_distance(bbox, img.shape)
    print(f'{p.name}: complete={ok_c} ({msg_c}) | dist={ok_d} ({msg_d})')

## Stage 3: Viewpoint Classification

In [ ]:
# ── Heuristic viewpoint check (shape-based, no classifier needed initially) ───
# A side-view cow has aspect ratio > 1.5 (wide body)
# A front/rear view has aspect ratio close to 1.0

def estimate_viewpoint_heuristic(bbox):
    """Quick viewpoint guess from bbox aspect ratio."""
    x1, y1, x2, y2 = bbox[:4]
    w, h = x2 - x1, y2 - y1
    ar = w / (h + 1e-6)
    if ar >= 1.6:
        return 'side', ar
    elif ar <= 0.8:
        return 'front_or_rear', ar
    else:
        return 'ambiguous', ar

# ── If dataset has viewpoint labels, train a proper classifier ────────────────
# Otherwise use the heuristic

# Test on detected images
for p in img_files[:10]:
    img = cv2.imread(str(p))
    if img is None: continue
    dets = detect_cows(img)
    if not dets: continue
    vp, ar = estimate_viewpoint_heuristic(dets[0])
    print(f'{p.name}: viewpoint={vp} (aspect_ratio={ar:.2f})')

In [ ]:
# ── Train lightweight viewpoint classifier if labels exist ────────────────────
# Check if viewpoint labels are present in the dataset
import json as _json
vp_labels = []
for jpath in ROOT.rglob('*.json'):
    try:
        with open(jpath) as f:
            data = _json.load(f)
        if isinstance(data, dict) and 'images' in data:
            for img_info in data['images']:
                vp = img_info.get('viewpoint', img_info.get('view', ''))
                if vp:
                    vp_labels.append(vp)
    except: pass

vp_counts = pd.Series(vp_labels).value_counts()
print('Viewpoint label distribution in dataset:')
print(vp_counts.to_string())

if len(vp_counts) < 2:
    print('\nInsufficient viewpoint labels — using heuristic aspect-ratio check.')
    USE_VIEWPOINT_CLASSIFIER = False
else:
    print('\nViewpoint labels found — training binary classifier...')
    USE_VIEWPOINT_CLASSIFIER = True

In [ ]:
# ── Viewpoint classifier (if labels exist) ────────────────────────────────────
if USE_VIEWPOINT_CLASSIFIER:
    from tensorflow.keras.applications import MobileNetV3Small
    from tensorflow.keras import layers, Model

    VP_SIZE = 128

    def build_viewpoint_classifier():
        inp = layers.Input(shape=(VP_SIZE, VP_SIZE, 3))
        base = MobileNetV3Small(include_top=False, weights='imagenet', input_tensor=inp,
                                  minimalistic=True)
        base.trainable = False
        x = layers.GlobalAveragePooling2D()(base.output)
        x = layers.Dense(64, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
        # Binary: side vs not-side
        out = layers.Dense(1, activation='sigmoid', name='is_side')(x)
        return Model(inputs=inp, outputs=out, name='viewpoint_clf')

    vp_model = build_viewpoint_classifier()
    print(vp_model.summary(line_length=80))

## Stage 4: Blur Detection

In [ ]:
def check_blur(img_bgr, bbox=None, threshold=80):
    """
    Laplacian variance blur check.
    Uses ROI from bbox if provided, else full image.
    Returns (ok, score, threshold).
    """
    if bbox is not None:
        x1, y1, x2, y2 = [max(0, int(v)) for v in bbox[:4]]
        roi = img_bgr[y1:y2, x1:x2]
    else:
        roi = img_bgr

    if roi.size == 0:
        return False, 0.0, threshold

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY) if roi.ndim == 3 else roi
    score = cv2.Laplacian(gray, cv2.CV_64F).var()
    return score >= threshold, score, threshold

# Calibrate threshold on dataset
blur_scores = []
for p in img_files[:200]:
    img = cv2.imread(str(p))
    if img is None: continue
    _, score, _ = check_blur(img)
    blur_scores.append(score)

arr = np.array(blur_scores)
print(f'Blur score distribution: min={arr.min():.0f} P5={np.percentile(arr,5):.0f} '
      f'median={np.median(arr):.0f} P95={np.percentile(arr,95):.0f} max={arr.max():.0f}')
print(f'Suggested threshold (P10): {np.percentile(arr, 10):.0f}')

# NOTE (notebook 01 finding): Training images score very high — mean≈2166, P25≈1139.
# These are professional 1900×1425 px photos; live camera frames will be blurrier.
# P5 of training data is a conservative real-world threshold; cap at 300 as a ceiling
# so the pipeline never blocks reasonable camera frames due to dataset bias.
BLUR_THRESHOLD = min(300, max(80, int(np.percentile(arr, 5))))
print(f'Using blur threshold: {BLUR_THRESHOLD}')
print('(P5 of dataset, capped at 300 to stay usable for live camera input)')

## Complete Pipeline — Frame Validator Function

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, Tuple, List

@dataclass
class FrameValidationResult:
    valid: bool
    stage_failed: Optional[str] = None
    user_message: Optional[str] = None
    cow_bbox: Optional[Tuple] = None
    n_cows: int = 0
    blur_score: float = 0.0
    viewpoint: str = ''
    distance_fraction: float = 0.0


def validate_frame(img_bgr,
                    detect_fn=detect_cows,
                    blur_thresh=BLUR_THRESHOLD) -> FrameValidationResult:
    """
    Full camera pipeline validation for one frame.
    Returns FrameValidationResult with:
      - valid=True  → safe to capture and run weight regression
      - valid=False → user guidance message for overlay
    """
    H, W = img_bgr.shape[:2]

    # Stage 1: Detection
    dets = detect_fn(img_bgr)
    n = len(dets)

    if n == 0:
        return FrameValidationResult(valid=False, n_cows=0,
            stage_failed='detection',
            user_message='No cow detected — point camera at the cow')

    if n > 1:
        return FrameValidationResult(valid=False, n_cows=n,
            stage_failed='multiple_cows',
            user_message=f'{n} cows detected — only one cow per measurement')

    bbox = dets[0]  # (x1, y1, x2, y2, score)
    x1, y1, x2, y2, conf = bbox

    # Stage 2: Completeness
    ok_c, msg_c = check_completeness(bbox, img_bgr.shape)
    if not ok_c:
        return FrameValidationResult(valid=False, n_cows=1, cow_bbox=bbox,
            stage_failed='completeness', user_message=msg_c)

    # Stage 3: Viewpoint
    vp, ar = estimate_viewpoint_heuristic(bbox)
    if vp != 'side':
        return FrameValidationResult(valid=False, n_cows=1, cow_bbox=bbox,
            stage_failed='viewpoint', viewpoint=vp,
            user_message='Capture from the side — rotate around the cow')

    # Stage 4: Blur
    ok_b, blur_score, _ = check_blur(img_bgr, bbox, threshold=blur_thresh)
    if not ok_b:
        return FrameValidationResult(valid=False, n_cows=1, cow_bbox=bbox,
            stage_failed='blur', blur_score=blur_score,
            user_message='Image blurry — hold phone steady')

    # Stage 5: Distance
    ok_d, msg_d = check_distance(bbox, img_bgr.shape)
    bbox_area_frac = (x2-x1)*(y2-y1) / (H*W)
    if not ok_d:
        return FrameValidationResult(valid=False, n_cows=1, cow_bbox=bbox,
            stage_failed='distance', distance_fraction=bbox_area_frac,
            user_message=msg_d)

    # All checks passed
    return FrameValidationResult(
        valid=True, n_cows=1, cow_bbox=bbox,
        blur_score=blur_score, viewpoint=vp,
        distance_fraction=bbox_area_frac,
        user_message='Hold steady — capturing...'
    )


# Test on sample images
print('Pipeline test results:')
for p in img_files[:10]:
    img = cv2.imread(str(p))
    if img is None: continue
    result = validate_frame(img)
    status = '✓ VALID' if result.valid else f'✗ {result.stage_failed}'
    print(f'{p.name}: {status} | {result.user_message}')

In [ ]:
# ── Pipeline pass-rate statistics ─────────────────────────────────────────────
pipeline_results = []
N_PIPE = min(300, len(img_files))
for p in tqdm(img_files[:N_PIPE], desc='Pipeline eval'):
    img = cv2.imread(str(p))
    if img is None: continue
    r = validate_frame(img)
    pipeline_results.append({
        'valid': r.valid,
        'stage_failed': r.stage_failed or 'PASS',
        'n_cows': r.n_cows,
    })

pipe_df = pd.DataFrame(pipeline_results)
print('Pipeline pass rate:', pipe_df['valid'].mean()*100, '%')
print('\nFailure stage distribution:')
print(pipe_df[~pipe_df['valid']]['stage_failed'].value_counts().to_string())

plt.figure(figsize=(8, 5))
pipe_df['stage_failed'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Frame Rejection Reasons')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('pipeline_rejection_reasons.png', dpi=120)
plt.show()

## Visualize Pipeline Overlay

In [ ]:
def draw_pipeline_overlay(img_bgr, result: FrameValidationResult) -> np.ndarray:
    """Draw guide overlay on frame (like the Android UI would show)."""
    overlay = img_bgr.copy()
    H, W = overlay.shape[:2]

    # Draw target zone rectangle
    margin = 0.10
    x1t = int(W * margin); y1t = int(H * margin)
    x2t = int(W * (1-margin)); y2t = int(H * (1-margin))
    color_zone = (0, 200, 0) if result.valid else (0, 100, 200)
    cv2.rectangle(overlay, (x1t, y1t), (x2t, y2t), color_zone, 2)

    # Draw detected bbox
    if result.cow_bbox:
        x1, y1, x2, y2 = [int(v) for v in result.cow_bbox[:4]]
        box_color = (0, 255, 0) if result.valid else (0, 0, 255)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), box_color, 3)

    # Status message
    msg = result.user_message or ''
    text_color = (0, 255, 0) if result.valid else (0, 80, 255)
    cv2.putText(overlay, msg, (20, H - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, text_color, 2, cv2.LINE_AA)

    # Stage indicator
    stages = ['detection', 'completeness', 'viewpoint', 'blur', 'distance']
    for i, stage in enumerate(stages):
        done = result.stage_failed is None or (
            stages.index(result.stage_failed) > i if result.stage_failed in stages else True
        )
        sc = (0, 200, 0) if done else (100, 100, 100)
        cv2.putText(overlay, f'[{"✓" if done else " "}] {stage}',
                    (10, 30 + i * 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, sc, 1)

    return overlay


# Show overlaid samples
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
shown = 0
for p in img_files:
    if shown >= 8: break
    img = cv2.imread(str(p))
    if img is None: continue
    r = validate_frame(img)
    vis = draw_pipeline_overlay(img, r)
    vis_rgb = cv2.cvtColor(cv2.resize(vis, (400, 300)), cv2.COLOR_BGR2RGB)
    ax = axes.flat[shown]
    ax.imshow(vis_rgb)
    ax.set_title('VALID' if r.valid else r.stage_failed, fontsize=10,
                  color='green' if r.valid else 'red')
    ax.axis('off')
    shown += 1

plt.suptitle('Camera Pipeline Overlay Simulation', fontsize=14)
plt.tight_layout()
plt.savefig('pipeline_overlay_demo.png', dpi=120)
plt.show()

## Android Inference — Kotlin Pseudocode

In [ ]:
kotlin_code = '''
// ──────────────────────────────────────────────────────────────
// CattleWeightInference.kt
// Drop cattle_weight_mobile_fp16.tflite into assets/
// ──────────────────────────────────────────────────────────────

class CattleWeightInference(context: Context) {

    // Normalization constants — from normalization_info.json (notebook 01 dataset):
    //   weight_mean = 163.3 kg  (dataset mean, n=26,943)
    //   weight_std  =  42.2 kg  (dataset std)
    private val WEIGHT_MEAN = 163.3f
    private val WEIGHT_STD  = 42.2f
    private val IMG_SIZE    = 224
    // AUX_DIM = 14 mask features + age_norm + sex = 16
    private val AUX_DIM     = 16

    // ImageNet normalization (matches albumentations Normalize in training)
    private val IMG_MEAN = floatArrayOf(0.485f, 0.456f, 0.406f)
    private val IMG_STD  = floatArrayOf(0.229f, 0.224f, 0.225f)

    private val interpreter: Interpreter

    init {
        val opts = Interpreter.Options().apply {
            numThreads = 4
            useNNAPI = true  // hardware acceleration
        }
        val model = loadModelFile(context, "cattle_weight_mobile_fp16.tflite")
        interpreter = Interpreter(model, opts)
    }

    /**
     * Predict weight from a side-view cow image + binary segmentation mask.
     *
     * @param bitmap   RGB bitmap (any size — will be resized to 224×224)
     * @param mask     Grayscale bitmap of same size (nullable — pass null if unavailable)
     * @param ageYears Age in years from cattle record (0.0f if unknown; dataset range ~1–10)
     * @param isMale   Sex flag — true for male/bull, false for female/heifer (false if unknown)
     * @return Estimated weight in kg (denormalized from model output)
     *
     * NOTE: Training data weight range was 50–621 kg (mean 163.3 kg). Predictions outside
     * this range may indicate poor image quality or an out-of-distribution animal.
     */
    fun predict(bitmap: Bitmap, mask: Bitmap?, ageYears: Float, isMale: Boolean): Float {
        // 1. Preprocess image
        val imgResized  = Bitmap.createScaledBitmap(bitmap, IMG_SIZE, IMG_SIZE, true)
        val maskResized = mask?.let { Bitmap.createScaledBitmap(it, IMG_SIZE, IMG_SIZE, true) }

        // 2. Build 4-channel float tensor [1, 224, 224, 4]  (RGB + mask channel)
        val imgTensor = Array(1) { Array(IMG_SIZE) { Array(IMG_SIZE) { FloatArray(4) } } }
        for (y in 0 until IMG_SIZE) {
            for (x in 0 until IMG_SIZE) {
                val pixel = imgResized.getPixel(x, y)
                val r = (Color.red(pixel)   / 255f - IMG_MEAN[0]) / IMG_STD[0]
                val g = (Color.green(pixel) / 255f - IMG_MEAN[1]) / IMG_STD[1]
                val b = (Color.blue(pixel)  / 255f - IMG_MEAN[2]) / IMG_STD[2]
                val m = if (maskResized != null) {
                    Color.red(maskResized.getPixel(x, y)) / 255f
                } else 0f
                imgTensor[0][y][x][0] = r
                imgTensor[0][y][x][1] = g
                imgTensor[0][y][x][2] = b
                imgTensor[0][y][x][3] = m
            }
        }

        // 3. Build aux vector [1, 16]
        //    Layout: [14 mask features] + [age_norm] + [sex]
        //    age_norm = ageYears / 10.0  (matches Python: age / 10.0, dataset range ~1–10 yr)
        val auxTensor = Array(1) { FloatArray(AUX_DIM) }
        val maskFeatures = if (maskResized != null)
            extractMaskFeatures(maskResized)  // 14-dim
        else FloatArray(14) { 0f }
        maskFeatures.copyInto(auxTensor[0])
        auxTensor[0][14] = ageYears / 10.0f
        auxTensor[0][15] = if (isMale) 1f else 0f

        // 4. Run inference
        val output = Array(1) { FloatArray(1) }
        val inputs  = mapOf("image" to imgTensor, "aux" to auxTensor)
        val outputs = mapOf(0 to output)
        interpreter.runSignature(inputs, outputs)

        // 5. Denormalize: output_norm * WEIGHT_STD + WEIGHT_MEAN
        val weightKg = output[0][0] * WEIGHT_STD + WEIGHT_MEAN
        return weightKg.coerceIn(40f, 750f)  // clamp to physiological range
    }

    /**
     * Extract 14-dim body measurement vector from mask bitmap.
     * Mirrors Python extract_mask_features_v2():
     *   [fill, w_norm, h_norm, aspect, solidity, compactness, extent,
     *    ellipse_ratio, skel_len, top_fill, bot_fill, perim_norm, hull_ratio, convexity_deficit]
     * In production use OpenCV4Android JNI for contour/ellipse calculations.
     */
    private fun extractMaskFeatures(mask: Bitmap): FloatArray {
        val gray = IntArray(IMG_SIZE * IMG_SIZE)
        mask.getPixels(gray, 0, IMG_SIZE, 0, 0, IMG_SIZE, IMG_SIZE)
        // Placeholder — implement same 14 features as Python extract_mask_features_v2
        // using OpenCV4Android: findContours, fitEllipse, arcLength, convexHull
        return FloatArray(14) { 0f }
    }

    private fun loadModelFile(ctx: Context, name: String): ByteBuffer {
        val afd = ctx.assets.openFd(name)
        val fis = FileInputStream(afd.fileDescriptor)
        val channel = fis.channel
        return channel.map(FileChannel.MapMode.READ_ONLY,
                            afd.startOffset, afd.declaredLength)
    }
}
'''

kotlin_path = WORK / 'CattleWeightInference.kt'
kotlin_path.write_text(kotlin_code)
print('Kotlin inference code saved to:', kotlin_path)
print(kotlin_code[:500])

In [ ]:
# ── Final deliverables checklist ──────────────────────────────────────────────
print('=' * 60)
print('FINAL DELIVERABLES CHECKLIST')
print('=' * 60)

deliverables = [
    ('01_dataset_exploration.ipynb', 'Complete dataset analysis'),
    ('02_model_training.ipynb', 'Architecture search + training'),
    ('03_camera_pipeline.ipynb', 'Camera validation pipeline'),
    ('cattle_weight_model.keras', 'Best teacher model (full precision)'),
    ('cattle_weight_fp16.tflite', 'Teacher TFLite FP16'),
    ('cattle_weight_mobile_fp16.tflite', 'Student mobile TFLite FP16'),
    ('normalization_info.json', 'Weight normalisation + model metadata'),
    ('CattleWeightInference.kt', 'Android inference Kotlin code'),
    ('error_analysis.png', 'Error analysis plots'),
    ('worst_predictions.png', 'Failure case visualizations'),
]

for fname, desc in deliverables:
    p = WORK / fname
    exists = p.exists() or Path(fname).exists()
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else '(notebook)'
    status = '✓' if exists else '○'
    print(f'  [{status}] {fname:<45} — {desc}  {size}')

print()
print('EXPECTED PERFORMANCE SUMMARY')
print('-' * 40)
print('  Teacher model MAPE :  ~6–8%')
print('  Student model MAPE :  +0.5–1.5% vs teacher')
print('  Mobile latency     :  150–250 ms on mid-range Android')
print('  Model size         :  < 20 MB (student FP16)')
print('  Memory footprint   :  < 30 MB')